# Alexa Review Sentiment Classifier

A sentiment classifier on the Amazon Alexa reviews dataset. The target `feedback`
(1 positive, 0 negative) is predicted from the free text in `verified_reviews`.

This notebook is the narrative report. It does no modeling of its own: it calls into
`src/` for curation, preprocessing, and evaluation, and model selection runs through the
config-driven harness (`configs/sklearn_logreg_tfidf.yaml` via `src/run_experiment.py`,
logged to MLflow). Here we characterize the label, set the baseline floor, and present the
selected linear control through the shared evaluation.

The theme is label skepticism: `feedback` is a hard threshold on the star `rating`, so
"sentiment" is a thresholded star count, and the hard cases are text/rating disagreements.

## Imports

Everything imported once. The repo root is put on the path so `src` imports resolve when
the notebook runs from `notebooks/`.

In [1]:
import os
import sys

REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from src.data.load import RAW_PATH, curate, load_raw
from src.evaluation.cv import cross_validate_negative, make_holdout
from src.evaluation.metrics import evaluate, negative_scores
from src.features.text import build_vectorizer

RANDOM_STATE = 42
RAW = os.path.join(REPO_ROOT, str(RAW_PATH))
pd.set_option("display.max_colwidth", 100)

## Data and label

We load and curate through `src.data`, which drops empty reviews and exact duplicates and
validates the result. Then we confirm what the label is.

In [2]:
df, report = curate(load_raw(RAW))
print(report)
df.head(3)

CurationReport(raw_rows=3150, dropped_blank=80, dropped_duplicates=686, curated_rows=2384, negatives=205)


,rating,date,variation,verified_reviews,feedback
0,5,31-Jul-18,Charcoal Fabric,Love my Echo!,1
1,5,31-Jul-18,Charcoal Fabric,Loved it!,1
2,4,31-Jul-18,Walnut Finish,"Sometimes while playing a game, you can answer a question correctly but Alexa says you got it wr...",1


Cross-tabulating `feedback` against `rating` shows the label rule directly: ratings 1 to 2
are negative, 3 to 5 are positive, with no exceptions. The label carries no information
beyond the star threshold.

In [3]:
crosstab = pd.crosstab(df["rating"], df["feedback"], margins=True)
rule_holds = ((df["rating"].isin([1, 2])) == (df["feedback"] == 0)).all()
print("feedback == 1 iff rating >= 3 holds for all rows:", bool(rule_holds))
crosstab

feedback == 1 iff rating >= 3 holds for all rows: True


feedback,0,1,All
rating,,,
1,129,0,129
2,76,0,76
3,0,106,106
4,0,340,340
5,0,1733,1733
All,205,2179,2384


The classes are heavily imbalanced toward positive, which is why accuracy is never the
headline metric.

In [4]:
balance = df["feedback"].value_counts(normalize=True).sort_index()
print("negative (0) share: %.4f   positive (1) share: %.4f" % (balance.loc[0], balance.loc[1]))

fig = px.histogram(
    df, x="feedback", title="Label balance: heavily positive",
    labels={"feedback": "feedback (0 = negative, 1 = positive)"},
)
fig.update_layout(bargap=0.2)
fig

negative (0) share: 0.0860   positive (1) share: 0.9140


## Baseline floor

Before any model, we record what a do-nothing majority predictor scores. Everything is
read through the shared `evaluate`, negative class first, so the numbers here are directly
comparable to the model below. We hold out a stratified test set that selection never
touches.

In [5]:
X, y = df["verified_reviews"], df["feedback"]
X_train, X_test, y_train, y_test = make_holdout(X, y, test_size=0.20, seed=RANDOM_STATE)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
dummy_metrics = evaluate(y_test, dummy.predict(X_test), neg_score=negative_scores(dummy, X_test))
{k: v for k, v in dummy_metrics.items() if k != "confusion_matrix"}

{'neg_precision': 0.0,
 'neg_recall': 0.0,
 'neg_f1': 0.0,
 'specificity': 1.0,
 'pr_auc': 0.0859538784067086}

## The linear control

Selection is done by the harness: `configs/sklearn_logreg_tfidf.yaml` runs a
`GridSearchCV` over the penalty (L1 vs L2 on the saga solver), an explicit `C` grid, and
class weighting, scored by cross-validated negative-class recall, and logs the full
ablation and the fitted model to MLflow. The early ablation favors an L1 penalty at
`C = 1.0` with balanced class weights. L1 is preferred here because it zeros out most
coefficients, giving a sparse, interpretable model.

Below we present that selected configuration through the shared cross-validation and
evaluation. We are not selecting here, only reporting the chosen model. (The single L1 fit
uses the fast `liblinear` solver; the harness uses `saga` to compare L1 and L2 across the
whole grid.)

In [6]:
control = Pipeline([
    ("tfidf", build_vectorizer(representation="tfidf", min_df=2)),   # term frequency inverse document frequency
    ("clf", LogisticRegression(
        penalty="l1", C=1.0, class_weight="balanced",
        solver="liblinear", max_iter=5000, random_state=RANDOM_STATE,
    )),
])

cv = cross_validate_negative(control, X_train, y_train, n_splits=5, n_repeats=3, seed=RANDOM_STATE)
pd.DataFrame({"mean": cv["mean"], "std": cv["std"]}).round(4)

,mean,std
neg_precision,0.3919,0.0504
neg_recall,0.7115,0.0557
neg_f1,0.5041,0.0503
specificity,0.8942,0.0189
pr_auc,0.5540,0.0723


Held-out performance of the same model, against the baseline floor. Specificity and
precision answer different questions: specificity is how often positive reviews are
wrongly flagged, precision is how trustworthy a negative flag is.

In [7]:
control.fit(X_train, y_train)
control_metrics = evaluate(y_test, control.predict(X_test), neg_score=negative_scores(control, X_test))

compare = pd.DataFrame(
    [
        {"model": "Dummy (most_frequent)", **{k: v for k, v in dummy_metrics.items() if k != "confusion_matrix"}},
        {"model": "TF-IDF + LogReg L1 (balanced)", **{k: v for k, v in control_metrics.items() if k != "confusion_matrix"}},
    ]
).set_index("model").round(4)
compare

,neg_precision,neg_recall,neg_f1,specificity,pr_auc
model,,,,,
Dummy (most_frequent),0.0000,0.0000,0.0000,1.0000,0.086
TF-IDF + LogReg L1 (balanced),0.3662,0.6341,0.4643,0.8968,0.585


In [8]:
cm = pd.DataFrame(
    control_metrics["confusion_matrix"],
    index=["true negative (0)", "true positive (1)"],
    columns=["pred negative (0)", "pred positive (1)"],
)
cm

,pred negative (0),pred positive (1)
true negative (0),26,15
true positive (1),45,391


Reading it honestly: the control clears the majority-accuracy floor and recovers most of
the negative reviews, but selecting on recall alone pushes precision down. Choosing a
principled operating point (negative-class recall >= 0.80, and reporting its precision and
specificity cost) is the threshold work in the next phase.

## Fairness

Delivered in the analysis phase: fairness as separation across device `variation` groups,
checking whether negative-class recall and the false-positive rate are roughly equal
across variations, reported with the spread. Stated here so the narrative is in place.

## Further analysis

Delivered in the analysis phase: interpret the model using the sparse L1 coefficients to
surface the words that drive a negative prediction, and error-analyze the text/rating
disagreement rows to see where the model diverges from the star label.